# 08.03 -- RLHF: The PPO Training Loop

**Module:** 08 -- Alignment  
**Notebook:** 3 of 5  
**Prerequisites:** 08.02 (Reward Models)

---

## Learning Objectives

1. Understand why reinforcement learning is necessary and what makes LM fine-tuning an RL problem
2. Derive the RLHF objective: reward maximisation with a KL divergence constraint
3. Understand PPO's role and why it was chosen for RLHF over simpler RL algorithms
4. Implement the full PPO training loop step by step using `trl`
5. Read and diagnose PPO training logs: what the reward, KL, and policy loss signals tell you

---

## 1. Why Reinforcement Learning?

Supervised fine-tuning (SFT) trains the model by maximum likelihood on fixed (prompt, response) pairs. This has a fundamental limitation: the training signal only tells the model to reproduce the training response, not to explore variations that might be even better.

Reinforcement learning changes the training setup. Instead of a fixed target response, the model generates its own outputs and receives a reward signal from the reward model. The policy (the language model being trained) is optimised to generate outputs that receive high rewards.

This is the key insight: RL allows the model to **discover** response strategies that the human annotators never explicitly demonstrated, as long as those strategies score well on the learned reward model.

---

## 2. The RLHF Objective

The formal objective being optimised is:

```
J(pi) = E_{x ~ D, y ~ pi(.|x)} [r(x, y)]  -  beta * KL[pi(.|x) || pi_ref(.|x)]

where:
  pi           = the policy (LM being trained)
  pi_ref       = the SFT reference policy (frozen)
  r(x, y)      = reward model score for response y to prompt x
  KL[pi||ref]  = KL divergence -- how far the policy has moved from the reference
  beta         = trade-off coefficient (typically 0.01 to 0.1)
```

The KL term is essential. Without it, the policy would exploit reward model weaknesses without constraint, collapsing into degenerate text that achieves high reward by gaming the proxy. The KL term is what prevents overoptimisation (see Notebook 08.01).

---

## 3. PPO in the RLHF Context

Proximal Policy Optimisation (PPO) is the RL algorithm used by InstructGPT and most production RLHF systems. PPO was chosen because:

1. **Stability**: PPO clips the policy update ratio, preventing catastrophically large updates that can destroy a policy in one bad batch -- critical when the policy is a 7B+ model that took weeks to train
2. **Sample efficiency**: PPO reuses each batch of generated samples for multiple gradient updates (unlike REINFORCE which requires fresh samples for every step)
3. **Established engineering**: PPO has extensive tooling and known hyperparameter ranges from robotics and game-playing literature

The PPO clipping objective is:

```
L_CLIP = E[min(
    ratio * advantage,
    clip(ratio, 1-eps, 1+eps) * advantage
)]

where ratio = pi(a|s) / pi_old(a|s)
```

Applied to language models, the 'action' is generating each token, the 'state' is the prompt and tokens generated so far, and the 'advantage' estimates how much better this response is than the expected response.

In [ ]:
# Environment setup.
#
# In addition to the standard libraries, this notebook requires trl >= 0.7
# for the PPOTrainer. The trl library implements the full RLHF loop including
# rollout generation, reward computation, advantage estimation, and the
# PPO update step. We also need peft for LoRA, which makes the policy
# memory-efficient during training.

!pip install transformers peft datasets trl>=0.7 accelerate matplotlib scipy -q

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from typing import List, Dict, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 4. The PPO Training Loop: Step by Step

The RLHF training loop has four repeating phases:

```
Phase 1: Rollout
  - Sample prompts from the dataset
  - Generate responses using the current policy (greedy or temperature sampling)

Phase 2: Reward
  - Score each (prompt, response) pair with the reward model
  - Compute the KL penalty: KL(pi || pi_ref) for each token
  - Final reward = reward_model_score - beta * KL_per_token.sum()

Phase 3: Advantage estimation
  - Use a value network to estimate the expected return from each state
  - Compute advantages: actual_return - estimated_value
  - Normalise advantages to stabilise gradients

Phase 4: PPO update
  - Apply the clipped PPO objective to update the policy
  - Update the value network towards the observed returns
  - Repeat phases 1-4 for the next batch
```

In [ ]:
# Implement the KL divergence penalty from scratch.
#
# The KL term is computed at the token level: for each token generated by
# the current policy, we compare its probability under the current policy
# vs the frozen reference (SFT) policy.
#
# KL(pi || pi_ref) = sum_t [ pi(a_t|s_t) * log(pi(a_t|s_t) / pi_ref(a_t|s_t)) ]
#
# In practice we compute the log-probability difference:
#   KL_approx = pi_log_prob - ref_log_prob
#
# This is the one-sample Monte Carlo estimate of KL divergence, which is
# what trl uses. It is unbiased but has higher variance than the exact KL.
# The variance is acceptable because we average over many tokens per batch.
#
# We visualise how different beta values affect the trade-off between
# reward maximisation and proximity to the reference policy.

def compute_kl_penalty(
    logprob_policy: torch.Tensor,  # (batch, seq_len) log probs under current policy
    logprob_ref:    torch.Tensor,  # (batch, seq_len) log probs under reference policy
    beta:           float = 0.05,  # KL penalty coefficient
) -> torch.Tensor:                 # (batch,) per-sequence KL penalty
    """
    Compute the per-sequence KL divergence penalty.

    The penalty is the sum of per-token log-probability differences,
    scaled by beta. A higher beta means the policy is penalised more
    for deviating from the reference.
    """
    # Token-level KL: log(pi/pi_ref) = log_pi - log_pi_ref
    kl_per_token = logprob_policy - logprob_ref  # (batch, seq_len)
    kl_per_seq   = kl_per_token.sum(dim=-1)      # (batch,)
    return beta * kl_per_seq


# Visualise: how does beta affect the effective reward signal?
# Scenario: reward model gives a score of 1.5, but the policy has drifted
# from the reference by different amounts (different KL values).

reward_model_score = 1.5
kl_values          = np.linspace(0, 10, 200)
betas              = [0.01, 0.05, 0.10, 0.20]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for beta in betas:
    effective_reward = reward_model_score - beta * kl_values
    ax.plot(kl_values, effective_reward, linewidth=2, label=f'beta={beta}')

ax.axhline(0, color='black', linestyle='--', alpha=0.4)
ax.axvline(reward_model_score / 0.05, color='gray', linestyle=':', alpha=0.6,
           label='Break-even KL (beta=0.05)')
ax.set_xlabel('KL Divergence from Reference Policy', fontsize=11)
ax.set_ylabel('Effective Reward (RM score - beta*KL)', fontsize=11)
ax.set_title('Effect of Beta on Effective Reward\n'
             'Higher beta penalises deviation from reference more strongly', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# PPO clipping illustration
ax = axes[1]
ratios     = np.linspace(0, 2.5, 300)
advantage  = 0.8   # positive advantage: this was a good action
epsilon    = 0.2   # PPO clipping parameter

unclipped  = ratios * advantage
clipped    = np.clip(ratios, 1 - epsilon, 1 + epsilon) * advantage
ppo_obj    = np.minimum(unclipped, clipped)

ax.plot(ratios, unclipped, '--', color='#1f77b4', linewidth=1.5, label='Unclipped (ratio * advantage)', alpha=0.7)
ax.plot(ratios, clipped,   '--', color='#d62728', linewidth=1.5, label='Clipped', alpha=0.7)
ax.plot(ratios, ppo_obj,   '-',  color='#2ca02c', linewidth=2.5, label='PPO objective (min of both)')
ax.axvline(1 - epsilon, color='gray', linestyle=':', alpha=0.6)
ax.axvline(1 + epsilon, color='gray', linestyle=':', alpha=0.6)
ax.axvspan(1 - epsilon, 1 + epsilon, alpha=0.07, color='green', label='Trust region [1-eps, 1+eps]')
ax.set_xlabel('Policy Ratio (pi_new / pi_old)', fontsize=11)
ax.set_ylabel('Objective Value', fontsize=11)
ax.set_title(f'PPO Clipped Objective (advantage={advantage}, epsilon={epsilon})\n'
             'Gradient is zero outside the trust region', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('RLHF Training Dynamics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/08_rlhf_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Full RLHF training loop using trl PPOTrainer.
#
# trl's PPOTrainer implements the complete RLHF pipeline:
#   1. It loads the policy model (the SFT model to be aligned)
#   2. It freezes a copy as the reference model
#   3. For each batch: generate responses, score them, compute KL penalty,
#      run PPO update steps
#
# The key arguments to understand in PPOConfig:
#   - kl_penalty: how the KL is computed ('kl' = token-level log-prob diff)
#   - init_kl_coef: the initial beta value for the KL penalty
#   - target_kl: if set, the KL coefficient is adapted to keep KL near this target
#   - ppo_epochs: number of PPO mini-batch updates per rollout batch
#   - mini_batch_size: number of examples per PPO update step
#   - cliprange: epsilon for PPO clipping (typically 0.1-0.3)
#   - vf_coef: coefficient for value function loss (typically 0.1)
#
# We demonstrate with GPT-2 as the policy, which is small enough to run
# on CPU for illustration. Production RLHF uses 7B+ models on multiple GPUs.

from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from trl import create_reference_model
from peft import LoraConfig, TaskType

MODEL_NAME = 'gpt2'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # PPO requires left-padding for generation

# AutoModelForCausalLMWithValueHead wraps a causal LM and adds a value head.
# The value head predicts the expected cumulative reward from each state,
# which is needed for advantage estimation in PPO.
# This is the only architecture difference from a standard language model.
policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME)
ref_model    = create_reference_model(policy_model)  # frozen copy for KL computation

ppo_config = PPOConfig(
    model_name=MODEL_NAME,
    learning_rate=1e-5,
    batch_size=4,            # number of prompts per rollout
    mini_batch_size=2,       # number per PPO update step
    ppo_epochs=2,            # number of PPO update passes over each rollout
    kl_penalty='kl',         # token-level log-probability KL
    init_kl_coef=0.05,       # initial beta for KL penalty
    target_kl=6.0,           # adaptive KL target (None to disable)
    cliprange=0.2,           # PPO epsilon
    cliprange_value=0.2,     # clipping for value function loss
    vf_coef=0.1,             # value function loss coefficient
    max_grad_norm=1.0,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
)

print("PPOTrainer initialised.")
print(f"Policy model parameters:    {sum(p.numel() for p in policy_model.parameters()):,}")
print(f"Value head parameters:      {sum(p.numel() for p in policy_model.v_head.parameters()):,}")
print(f"Reference model (frozen):   {sum(p.numel() for p in ref_model.parameters()):,}")

In [ ]:
# Simple proxy reward function for the demonstration.
#
# In a real RLHF run, this function calls the trained reward model from
# Notebook 08.02. Here we use a lightweight proxy that rewards responses
# containing technical vocabulary, appropriate length, and correct format.
#
# The proxy is intentionally imperfect -- it will be exploited by the policy
# to some degree, which is the expected behaviour and the reason real systems
# use large, carefully trained reward models with multiple heads.
#
# A real reward function signature is:
#   def reward_fn(prompt_texts, response_texts) -> list[float]
# where each element is the scalar reward for one (prompt, response) pair.

TECHNICAL_TERMS = {
    'gradient', 'backpropagation', 'loss', 'parameter', 'weight',
    'layer', 'neuron', 'activation', 'learning', 'training',
    'epoch', 'batch', 'optimisation', 'regularisation', 'overfitting',
    'attention', 'transformer', 'embedding', 'softmax', 'normalisation',
}

def proxy_reward(prompt_texts: List[str], response_texts: List[str]) -> List[float]:
    """
    Compute proxy rewards for a batch of (prompt, response) pairs.

    Rewards are in the range [0, 1]. The proxy combines:
    - Technical vocabulary coverage (0.5 weight)
    - Response length in a preferred range (0.3 weight)
    - Ending with a complete sentence (0.2 weight)
    """
    rewards = []
    for prompt, response in zip(prompt_texts, response_texts):
        words = response.lower().split()
        n_words = len(words)

        # Technical vocabulary score: how many domain terms appear?
        term_hits = sum(1 for w in words if w.strip('.,;:') in TECHNICAL_TERMS)
        vocab_score = min(1.0, term_hits / 5)

        # Length score: target 30-100 words
        if n_words < 10:
            length_score = n_words / 10
        elif n_words <= 100:
            length_score = 1.0
        else:
            length_score = max(0.3, 1.0 - (n_words - 100) / 300)

        # Format score: ends with sentence-terminating punctuation
        format_score = 1.0 if response.strip().endswith(('.', '!', '?')) else 0.5

        reward = 0.5 * vocab_score + 0.3 * length_score + 0.2 * format_score
        rewards.append(reward)

    return rewards


# Test the reward function on sample responses
test_cases = [
    ("What is backpropagation?", "Backpropagation computes gradients by applying the chain rule backwards through the loss landscape, updating each weight proportionally to its contribution to the error."),
    ("What is backpropagation?", "It is a thing."),
    ("What is backpropagation?", "Backpropagation. Backpropagation. Backpropagation. " * 30),  # reward hacking attempt
]

prompts   = [t[0] for t in test_cases]
responses = [t[1] for t in test_cases]
test_rewards = proxy_reward(prompts, responses)

print("Proxy reward sanity check:")
for (p, r), score in zip(test_cases, test_rewards):
    print(f"  reward={score:.3f}  response='{r[:70]}'")

In [ ]:
# Run the PPO training loop.
#
# Each iteration of the outer loop is one PPO 'epoch' over one batch of prompts.
# The inner structure handled by ppo_trainer.step() is:
#   1. Compute log-probabilities under the reference model for each response
#      (needed for the KL penalty)
#   2. Compute advantages using the value head predictions
#   3. Run ppo_config.ppo_epochs mini-batch PPO updates
#   4. Return statistics for logging
#
# Key statistics to monitor during training:
#   - ppo/mean_scores: mean reward model score (should increase)
#   - ppo/mean_non_score_reward: the KL penalty term (should stay bounded)
#   - ppo/policy/approxkl: approximate KL divergence from reference
#   - ppo/loss/policy: PPO policy gradient loss
#   - ppo/loss/value: value function loss (should decrease as VF improves)
#
# We run 10 iterations for demonstration. A production RLHF run typically
# runs for thousands of iterations on tens of thousands of prompts.

TRAINING_PROMPTS = [
    "Explain what a learning rate is in neural network training.",
    "What is the purpose of a loss function?",
    "Describe how attention mechanisms work in transformers.",
    "What is the difference between training and inference?",
    "Why do we use batch normalisation?",
    "What is gradient descent?",
    "Explain overfitting and how to prevent it.",
    "What is the role of activation functions?",
]

N_ITERATIONS = 10
BATCH_SIZE   = 4
GEN_KWARGS   = dict(
    max_new_tokens=80,
    do_sample=True,
    temperature=0.9,
    pad_token_id=tokenizer.eos_token_id,
)

training_stats = defaultdict(list)

import random
random.seed(42)

from collections import defaultdict

for iteration in range(N_ITERATIONS):
    # Sample a batch of prompts
    batch_prompts = random.choices(TRAINING_PROMPTS, k=BATCH_SIZE)

    # Tokenise prompts. PPOTrainer expects a list of 1-D tensors.
    prompt_tensors = [
        tokenizer(p, return_tensors='pt')['input_ids'].squeeze(0)
        for p in batch_prompts
    ]

    # Generate responses using the current policy
    response_tensors = ppo_trainer.generate(
        prompt_tensors,
        return_prompt=False,
        **GEN_KWARGS,
    )

    # Decode responses to text for reward computation
    response_texts = [
        tokenizer.decode(r, skip_special_tokens=True)
        for r in response_tensors
    ]

    # Compute rewards (list of scalar tensors, one per example)
    raw_rewards = proxy_reward(batch_prompts, response_texts)
    reward_tensors = [torch.tensor(r) for r in raw_rewards]

    # Run the PPO update step.
    # ppo_trainer.step() handles:
    #   - Reference model log-prob computation
    #   - KL penalty computation and addition to rewards
    #   - Advantage estimation via the value head
    #   - Multiple PPO mini-batch updates
    stats = ppo_trainer.step(prompt_tensors, response_tensors, reward_tensors)

    # Record key statistics for later analysis
    training_stats['reward'].append(np.mean(raw_rewards))
    training_stats['kl'].append(stats.get('ppo/mean_non_score_reward', 0))
    training_stats['policy_loss'].append(stats.get('ppo/loss/policy', 0))
    training_stats['value_loss'].append(stats.get('ppo/loss/value', 0))

    if (iteration + 1) % 2 == 0:
        print(f"Iter {iteration+1:>3}/{N_ITERATIONS}  "
              f"reward={np.mean(raw_rewards):.3f}  "
              f"kl_penalty={stats.get('ppo/mean_non_score_reward', 0):.3f}  "
              f"policy_loss={stats.get('ppo/loss/policy', 0):.4f}")

print("\nPPO training complete.")

In [ ]:
# Visualise PPO training dynamics.
#
# The four panels show the key signals to monitor during RLHF training.
# Understanding what each signal means allows you to diagnose problems early.
#
# Reward: should increase as the policy learns to generate better responses.
#   If it increases very fast then plateaus, the policy may be reward hacking.
#
# KL penalty: the negative contribution of the KL term to the effective reward.
#   Should be non-zero and bounded. If it grows continuously, the policy is
#   drifting far from the reference, increasing overoptimisation risk.
#
# Policy loss: the PPO gradient loss. Should oscillate around a small value.
#   Large spikes indicate instability; try reducing the learning rate.
#
# Value loss: the critic's loss. Should decrease as the value head learns to
#   predict returns accurately. A value head that does not converge leads to
#   poor advantage estimates and slow policy learning.

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
iterations_axis = range(1, N_ITERATIONS + 1)

panel_configs = [
    ('reward',       'Mean Reward',       '#2ca02c', 'Should increase during training'),
    ('kl',           'KL Penalty',        '#d62728', 'Should stay bounded (not diverge)'),
    ('policy_loss',  'PPO Policy Loss',   '#1f77b4', 'Should oscillate near a small value'),
    ('value_loss',   'Value Head Loss',   '#ff7f0e', 'Should decrease as value head converges'),
]

for ax, (key, label, color, note) in zip(axes.flat, panel_configs):
    values = training_stats[key]
    if values:
        ax.plot(list(iterations_axis)[:len(values)], values,
                'o-', color=color, linewidth=2, markersize=5)
    ax.set_xlabel('Iteration', fontsize=10)
    ax.set_ylabel(label, fontsize=10)
    ax.set_title(f'{label}\n({note})', fontsize=11)
    ax.grid(alpha=0.3)

plt.suptitle('PPO Training Dashboard -- Key Monitoring Signals', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/08_ppo_training.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare responses from the base model vs the PPO-aligned model.
#
# This is the qualitative evaluation step. After training, we generate
# responses to held-out prompts (not seen during training) using both
# the original base model and the PPO-aligned policy.
#
# We expect the aligned model to produce responses that score higher on
# the proxy reward, which in this case means more technical vocabulary,
# better length, and cleaner sentence endings.
#
# In a real evaluation, this comparison would be done by human raters
# or an LLM judge (as implemented in Notebook 06.03).

from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained('gpt2')
base_model.eval()

eval_prompts = [
    "What is the purpose of dropout in neural networks?",
    "Explain what an embedding layer does.",
]

def generate_response(model, tokenizer, prompt: str, max_new_tokens: int = 80) -> str:
    inputs = tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Qualitative comparison: Base model vs PPO-aligned model")
print("=" * 68)

for prompt in eval_prompts:
    base_resp = generate_response(base_model, tokenizer, prompt)
    aligned_resp = generate_response(policy_model.pretrained_model, tokenizer, prompt)

    base_reward    = proxy_reward([prompt], [base_resp])[0]
    aligned_reward = proxy_reward([prompt], [aligned_resp])[0]

    print(f"\nPrompt: {prompt}")
    print(f"  Base model (reward={base_reward:.3f}):")
    print(f"    {base_resp[:150]}")
    print(f"  Aligned model (reward={aligned_reward:.3f}):")
    print(f"    {aligned_resp[:150]}")
    print(f"  Reward delta: {aligned_reward - base_reward:+.3f}")

## 5. Engineering Notes

**PPO hyperparameter guide for RLHF**

| Hyperparameter | Typical range | What it controls |
|---------------|--------------|------------------|
| `init_kl_coef` (beta) | 0.01 -- 0.2 | Strength of KL penalty; higher = more conservative |
| `target_kl` | 3 -- 10 | Adaptive KL target; set to None to fix beta |
| `ppo_epochs` | 1 -- 4 | Mini-batch updates per rollout; higher = more efficient but less stable |
| `cliprange` | 0.1 -- 0.3 | PPO trust region size; smaller = more conservative |
| `learning_rate` | 1e-6 -- 1e-4 | Lower than SFT; RLHF is more sensitive to LR |
| `batch_size` | 32 -- 256 | Number of prompts per rollout; larger = lower gradient variance |
| `vf_coef` | 0.05 -- 0.5 | Value function loss weight in total loss |

**Signs that PPO training is going wrong**

| Symptom | Likely cause | Remedy |
|---------|-------------|--------|
| KL diverges (keeps increasing) | beta too low | Increase `init_kl_coef` or lower LR |
| Reward collapses after initial increase | Reward hacking | Improve reward model; add more human preference data |
| Policy loss has large spikes | LR too high or cliprange too large | Reduce LR; reduce `cliprange` |
| Value loss stays high | Value head not converging | Increase `vf_coef`; train value head more |
| Response quality degrades | Overoptimisation | Reduce `ppo_epochs`; increase KL coefficient |

## 6. Exercises

1. **Beta sensitivity**: Run PPO with three beta values: 0.01, 0.05, and 0.2. Plot the KL divergence curve for each. At what beta does the policy stay nearest to the reference while still improving the reward?

2. **Reward hacking in practice**: Design a proxy reward that incentivises long responses (e.g., `reward = min(1.0, n_words / 200)`). Run PPO for 50 iterations and plot response length over time. How quickly does the policy learn to game it?

3. **Adaptive KL vs fixed beta**: Compare training with `target_kl=6.0` (adaptive) vs `target_kl=None, init_kl_coef=0.05` (fixed). Plot the KL divergence over time. Which produces more stable training?

4. **Value head initialisation**: Train two PPO runs -- one with the value head initialised from zero weights and one from small random weights. Plot the value loss over the first 20 iterations. Which converges faster?

5. **Rollout quality vs quantity**: Compare `batch_size=4` (many small rollout batches) vs `batch_size=32` (few large rollout batches) for the same total number of training examples. Which produces lower variance reward estimates?

---

## 7. References

- [Schulman et al. (2017) -- Proximal Policy Optimization Algorithms](https://arxiv.org/abs/1707.06347)
- [Ouyang et al. (2022) -- Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155)
- [Ziegler et al. (2019) -- Fine-Tuning Language Models from Human Preferences](https://arxiv.org/abs/1909.08593)
- [trl PPOTrainer Documentation](https://huggingface.co/docs/trl/ppo_trainer)